# 01 — Data Profiling & Cleaning

| Section | What happens | Tables involved |
|---------|-------------|----------------|
| **Profiling** | Observe only — no changes | all 5 |
| **Cleaning** | `basic_cleaning`, fuzzy match, `fix_unit_errors`, `standardize_quantities` | benchmark, inventory, invoices, recipes |
| **Food-only filter** | convert olio bt→lt, then exclude beverage rows | inventory, sales_pos, invoices |
| **Outlier fixing** | detect (multiplier=4) → NaN → benchmark fill | inventory, invoices |
| **Augmentation** | hardcoded dict → `pd.concat` | recipes only |

> Food-only filter runs **after** `standardize_quantities` so all unit variants (bt, 750ml, bott…) are already normalized before exclusion.

---
## Setup

In [ ]:
%load_ext autoreload
%autoreload 2

import os
import sys
sys.path.append(os.path.join('..', 'src'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import utils as ut

RAW = os.path.join('..', 'data', 'raw')
print('Setup complete.')

In [ ]:
# --- CONSTANTS (calibrated from ebitda_pipeline) ---

PREMIUM_CATEGORIES = ['zafferano', 'tartufo', 'caviale', 'vaniglia']
SAFE_CATEGORIES    = ['carni', 'verdure', 'latticini']

MAPPING_DICT = {
    'olio ev':              'olio extravergine oliva',
    'olio evo':             'olio extravergine oliva',
    'olio evo lt':          'olio extravergine oliva',
    'sale 100/kg':          'sale fino',
    'uova 30pz':            'uova fresche',
    'uova xl':              'uova fresche',
    'fil. manzo':           'filetto di manzo',
    'salmone inter':        'salmone',
    'salmone intero':       'salmone',
    'salmoneo':             'salmone',
    'parm.regg.':           'parmigiano reggiano',
}

CONVERSION_MAP = {
    'burrata':              (0.125, 'pz', 'kg'),
    'uova':                 (0.060, 'pz', 'kg'),
    'panna fresca liquida': (1.0,   ['bt', '750ml', '0.75'], 'lt'),
}

print('Constants defined.')

In [ ]:
# --- FOOD-ONLY FILTER CONSTANTS ---
# Applied AFTER standardize_quantities so all unit variants are normalized

BEV_CATEGORIES = {'bevande', 'vini', 'liquori', 'birre'}
BEV_UNITS      = {'bt', 'cl', 'bottiglia'}   # 'bt' catches 750ml/bott post-standardize
BEV_KEYWORDS   = ['vino', 'birra', 'acqua', 'coca', 'prosecco', 'champagne', 'spirits']
BEV_PATTERN    = '|'.join(BEV_KEYWORDS)

print('Food-only filter constants defined.')

In [ ]:
benchmark = pd.read_csv(os.path.join(RAW, 'benchmark_ingredienti_horeca.csv'), low_memory=False)
inventory  = pd.read_csv(os.path.join(RAW, 'inventory_stock.csv'),             low_memory=False)
sales_pos  = pd.read_csv(os.path.join(RAW, 'raw_sales_pos.csv'),               low_memory=False)
invoices   = pd.read_csv(os.path.join(RAW, 'supplier_invoices.csv'),           low_memory=False)
recipes    = pd.read_csv(os.path.join(RAW, 'recipe_book_unstandardized.csv'),  low_memory=False)

print('Files loaded:')
for name, df in [('benchmark', benchmark), ('inventory', inventory),
                 ('sales_pos', sales_pos), ('invoices', invoices), ('recipes', recipes)]:
    print(f'  {name:<12} {df.shape[0]:>6} rows  x  {df.shape[1]} cols')

---
# 1. PROFILING
Observe the raw state of each table — no modifications.

### 1.1 Benchmark

In [ ]:
print('--- BENCHMARK ---')
display(benchmark.head())
ut.dupli_nan_count(benchmark)
print()
_ = ut.outliers_auto_detection(benchmark)

### 1.2 Inventory

In [ ]:
print('--- INVENTORY ---')
display(inventory.head())
ut.dupli_nan_count(inventory)
print()
ut.date_accuracy(inventory.copy(), 'data_rilevazione', [2023, 2024])
print()
_ = ut.outliers_auto_detection(inventory)

### 1.3 Sales POS

In [ ]:
print('--- SALES POS ---')
display(sales_pos.head())
ut.dupli_nan_count(sales_pos)
print()
ut.date_accuracy(sales_pos.copy(), 'data_ora', [2023, 2024])
print()
_ = ut.outliers_auto_detection(sales_pos)
print()
print('Category breakdown (incl. beverages):')
display(sales_pos['categoria'].value_counts(dropna=False))

### 1.4 Invoices

In [ ]:
print('--- INVOICES ---')
display(invoices.head())
ut.dupli_nan_count(invoices)
print()
ut.date_accuracy(invoices.copy(), 'data_fattura', [2023, 2024])
print()
_ = ut.outliers_auto_detection(invoices)
print()
print('Unit of measurement breakdown:')
display(invoices['unita_misura'].value_counts(dropna=False))

### 1.5 Recipes

In [ ]:
print('--- RECIPES ---')
display(recipes)
ut.dupli_nan_count(recipes)
print()
_ = ut.outliers_auto_detection(recipes)

### 1.6 Summary table

In [ ]:
summary_rows = []
for name, df in [('benchmark', benchmark), ('inventory', inventory),
                 ('sales_pos', sales_pos), ('invoices', invoices), ('recipes', recipes)]:
    summary_rows.append({
        'file':          name,
        'rows':          df.shape[0],
        'cols':          df.shape[1],
        'duplicates':    df.duplicated().sum(),
        'avg_missing_%': round(df.isna().mean().mean() * 100, 1),
    })
display(pd.DataFrame(summary_rows))

---
# 2. CLEANING
Order per table: `basic_cleaning` → fuzzy match → `fix_unit_errors` → `standardize_quantities` → `quantity_exception_manage` → **food-only filter**.

Raw tables are never overwritten.

### 2.1 Benchmark

In [ ]:
benchmark_cleaned = ut.basic_cleaning(benchmark, MAPPING_DICT)

# Price lookup dict — shared across all outlier fixing steps
benchmark_dict = dict(zip(
    benchmark_cleaned['Ingrediente_Standard'],
    benchmark_cleaned['Prezzo_Benchmark_EUR']
))

print(f'benchmark_cleaned: {benchmark_cleaned.shape}')
display(benchmark_cleaned.head())

### 2.2 Inventory

In [ ]:
# Step 1 — text normalization + date parsing
inventory_cleaned = ut.basic_cleaning(inventory, MAPPING_DICT)
display(inventory_cleaned.head())

In [ ]:
# Step 2 — fuzzy match → canonical ingredient names
fuzzy_map_inventory = ut.get_best_match(
    inventory_cleaned['prodotto'],
    benchmark_cleaned['Ingrediente_Standard'],
    threshold=70
)
inventory_cleaned['nome_standard'] = inventory_cleaned['prodotto'].map(fuzzy_map_inventory)

unmatched_inv = inventory_cleaned.loc[inventory_cleaned['nome_standard'].isna()]
print(f'Unmatched: {unmatched_inv["prodotto"].nunique()} unique names')

# NaN Handling Rule — print breakdown before any action
# AWAIT GIOVANNI APPROVAL
print('\n--- NaN nome_standard — inventory ---')
print(f'Total NaN rows  : {len(unmatched_inv)}')
print(f'Unique names    : {unmatched_inv["prodotto"].nunique()}')
print('\nCategoria breakdown:')
display(unmatched_inv['categoria'].value_counts(dropna=False))
print('\nUnmatched products:')
display(unmatched_inv.drop_duplicates(subset=['prodotto'])[['prodotto', 'categoria']])

In [ ]:
# Step 3 — fix g/kg typos
inventory_fixed = ut.fix_unit_errors(
    inventory_cleaned,
    unit_col='unita_misura', price_col='valore_unitario',
    item_col='nome_standard', cat_col='categoria',
    threshold=10,
    premium_list=PREMIUM_CATEGORIES, safe_cats=SAFE_CATEGORIES
)

# Edge case: 'cassa' of eggs = 1.8 kg
cassa_mask = inventory_fixed['unita_misura'] == 'cassa'
inventory_fixed.loc[cassa_mask, ['qty_teorica', 'qty_fisica']] *= 1.8
inventory_fixed.loc[cassa_mask, 'valore_unitario'] /= 1.8
inventory_fixed.loc[cassa_mask, 'unita_misura'] = 'kg'
print(f'Cassa rows converted: {cassa_mask.sum()}')

In [ ]:
# Step 4 — normalize units to kg / lt
inventory_cleaned_std = ut.standardize_quantities(
    inventory_fixed, qty_cols=['qty_fisica', 'qty_teorica'], unit_col='unita_misura'
)

# Step 5 — piece-to-weight conversions
for qty_col in ['qty_fisica', 'qty_teorica']:
    inventory_cleaned_std = ut.quantity_exception_manage(
        inventory_cleaned_std,
        product_col='nome_standard', quantity_col=qty_col,
        uom_col='unita_misura', conversion_map=CONVERSION_MAP,
        price_col='valore_unitario'
    )

print('Units before food filter:', inventory_cleaned_std['unita_misura'].value_counts().to_dict())

#### Food-only filter (post-standardize)

In [ ]:
# Rule 1 — CONVERT: olio in 'bt' → lt (1 bt = 0.75 lt)
# Check both prodotto and nome_standard to catch all variants
olio_bt_mask = (
    (inventory_cleaned_std['prodotto'].str.contains('olio', na=False) |
     inventory_cleaned_std['nome_standard'].str.contains('olio', na=False)) &
    (inventory_cleaned_std['unita_misura'] == 'bt')
)
inventory_cleaned_std.loc[olio_bt_mask, 'qty_fisica']      *= 0.75
inventory_cleaned_std.loc[olio_bt_mask, 'qty_teorica']     *= 0.75
inventory_cleaned_std.loc[olio_bt_mask, 'valore_unitario'] /= 0.75
inventory_cleaned_std.loc[olio_bt_mask, 'unita_misura']    = 'lt'
print(f'Olio bt→lt: {olio_bt_mask.sum()} rows converted')

# Rule 2 — EXCLUDE
n_before  = len(inventory_cleaned_std)
cat_mask  = inventory_cleaned_std['categoria'].isin(BEV_CATEGORIES)
unit_mask = inventory_cleaned_std['unita_misura'].isin(BEV_UNITS)
kw_mask   = inventory_cleaned_std['prodotto'].str.contains(BEV_PATTERN, na=False)
excl_mask = cat_mask | unit_mask | kw_mask

inventory_cleaned_std = inventory_cleaned_std[~excl_mask].copy()
n_after   = len(inventory_cleaned_std)
print(f'inventory  | before: {n_before:>5} | after: {n_after:>5} | excluded: {n_before - n_after}')

# Final enforcement — drop anything still not in ('kg', 'lt')
non_std = inventory_cleaned_std[~inventory_cleaned_std['unita_misura'].isin(['kg', 'lt'])]
if len(non_std):
    print(f'\nEnforcement drop — {len(non_std)} rows with unita_misura not in (kg, lt):')
    print(non_std[['prodotto', 'categoria', 'unita_misura', 'qty_fisica']].to_string())
    inventory_cleaned_std = inventory_cleaned_std[inventory_cleaned_std['unita_misura'].isin(['kg', 'lt'])].copy()
else:
    print('Enforcement OK — only kg/lt remain.')
print('Units after filter:', inventory_cleaned_std['unita_misura'].value_counts().to_dict())

### 2.3 Sales POS

In [ ]:
# Step 1 — text normalization + date parsing
sales_pos_cleaned = ut.basic_cleaning(sales_pos, MAPPING_DICT)

# Step 2 — drop date outliers
sales_pos_cleaned['anno'] = pd.to_datetime(
    sales_pos_cleaned['data_ora'], format='mixed', dayfirst=True
).dt.year
n_before = len(sales_pos_cleaned)
sales_pos_cleaned = sales_pos_cleaned[sales_pos_cleaned['anno'].isin([2023, 2024])]
print(f'Date outliers dropped: {n_before - len(sales_pos_cleaned)} rows')

# Step 3 — food-only filter
# 'extra' category = coperti (covers charge) — excluded from food pipeline
# sales_pos has no unita_misura → filter on categoria + item_name keywords only
SALES_EXCL_CATEGORIES = BEV_CATEGORIES | {'extra'}

n_before  = len(sales_pos_cleaned)
cat_mask  = sales_pos_cleaned['categoria'].isin(SALES_EXCL_CATEGORIES)
kw_mask   = sales_pos_cleaned['item_name'].str.contains(BEV_PATTERN, na=False)
excl_mask = cat_mask | kw_mask

sales_pos_cleaned = sales_pos_cleaned[~excl_mask].copy()
n_after   = len(sales_pos_cleaned)
print(f'sales_pos  | before: {n_before:>5} | after: {n_after:>5} | excluded: {n_before - n_after}')

print(f'\nsales_pos_cleaned: {sales_pos_cleaned.shape}')
display(sales_pos_cleaned.head())

### 2.4 Invoices

In [ ]:
# Step 1 — text normalization + date parsing
invoices_cleaned = ut.basic_cleaning(invoices, MAPPING_DICT)
display(invoices_cleaned.head())

In [ ]:
# Step 2 — fuzzy match
fuzzy_map_invoices = ut.get_best_match(
    invoices_cleaned['descrizione_prodotto'],
    benchmark_cleaned['Ingrediente_Standard'],
    threshold=70
)
invoices_cleaned['nome_standard'] = invoices_cleaned['descrizione_prodotto'].map(fuzzy_map_invoices)

unmatched_inv2 = invoices_cleaned.loc[invoices_cleaned['nome_standard'].isna()]

# NaN Handling Rule — print breakdown before any action
# AWAIT GIOVANNI APPROVAL
print('\n--- NaN nome_standard — invoices ---')
print(f'Total NaN rows  : {len(unmatched_inv2)}')
print(f'Unique names    : {unmatched_inv2["descrizione_prodotto"].nunique()}')
print('\nCategoria breakdown:')
display(unmatched_inv2['categoria'].value_counts(dropna=False))
print('\nUnmatched products (sorted by price):')
display(
    unmatched_inv2[['categoria', 'descrizione_prodotto', 'unita_misura', 'prezzo_unitario', 'nome_standard']]
    .sort_values('prezzo_unitario', ascending=False)
    .drop_duplicates(subset=['descrizione_prodotto'])
)

In [ ]:
# Step 3 — fix g/kg typos
invoices_fixed = ut.fix_unit_errors(
    invoices_cleaned,
    unit_col='unita_misura', price_col='prezzo_unitario',
    item_col='descrizione_prodotto', cat_col='categoria',
    threshold=10,
    premium_list=PREMIUM_CATEGORIES, safe_cats=SAFE_CATEGORIES
)

In [ ]:
# Step 4 — normalize units
invoices_cleaned_std = ut.standardize_quantities(
    invoices_fixed, qty_cols='quantita', unit_col='unita_misura'
)

# Step 5 — piece-to-weight conversions (panna bt→lt, burrata pz→kg, uova pz→kg)
invoices_cleaned_std = ut.quantity_exception_manage(
    invoices_cleaned_std,
    product_col='nome_standard', quantity_col='quantita',
    uom_col='unita_misura', conversion_map=CONVERSION_MAP,
    price_col='prezzo_unitario'
)

print('Units before food filter:', invoices_cleaned_std['unita_misura'].value_counts().to_dict())

#### Food-only filter (post-standardize)

In [ ]:
# Rule 1 — CONVERT: olio in 'bt' → lt (1 bt = 0.75 lt)
# Check both descrizione_prodotto and nome_standard to catch all variants
olio_bt_mask2 = (
    (invoices_cleaned_std['descrizione_prodotto'].str.contains('olio', na=False) |
     invoices_cleaned_std['nome_standard'].str.contains('olio', na=False)) &
    (invoices_cleaned_std['unita_misura'] == 'bt')
)
invoices_cleaned_std.loc[olio_bt_mask2, 'quantita']        *= 0.75
invoices_cleaned_std.loc[olio_bt_mask2, 'prezzo_unitario'] /= 0.75
invoices_cleaned_std.loc[olio_bt_mask2, 'unita_misura']    = 'lt'
print(f'Olio bt→lt: {olio_bt_mask2.sum()} rows converted')

# Rule 2 — EXCLUDE
n_before  = len(invoices_cleaned_std)
cat_mask  = invoices_cleaned_std['categoria'].isin(BEV_CATEGORIES)
unit_mask = invoices_cleaned_std['unita_misura'].isin(BEV_UNITS)
kw_mask   = invoices_cleaned_std['descrizione_prodotto'].str.contains(BEV_PATTERN, na=False)
excl_mask = cat_mask | unit_mask | kw_mask

invoices_cleaned_std = invoices_cleaned_std[~excl_mask].copy()
n_after   = len(invoices_cleaned_std)
print(f'invoices   | before: {n_before:>5} | after: {n_after:>5} | excluded: {n_before - n_after}')

# Recalculate total amount on corrected prices
invoices_cleaned_std['importo_totale_pulito'] = (
    invoices_cleaned_std['prezzo_unitario'] * invoices_cleaned_std['quantita']
).round(2)

# Final enforcement — drop anything still not in ('kg', 'lt')
non_std = invoices_cleaned_std[~invoices_cleaned_std['unita_misura'].isin(['kg', 'lt'])]
if len(non_std):
    print(f'\nEnforcement drop — {len(non_std)} rows with unita_misura not in (kg, lt):')
    print(non_std[['descrizione_prodotto', 'categoria', 'unita_misura', 'quantita']].to_string())
    invoices_cleaned_std = invoices_cleaned_std[invoices_cleaned_std['unita_misura'].isin(['kg', 'lt'])].copy()
else:
    print('Enforcement OK — only kg/lt remain.')
print('Units after filter:', invoices_cleaned_std['unita_misura'].value_counts().to_dict())

### 2.5 Recipes

In [ ]:
recipes_cleaned = ut.basic_cleaning(recipes, MAPPING_DICT)

recipes_cleaned_std = ut.standardize_quantities(
    recipes_cleaned, qty_cols='quantita_per_porzione', unit_col='unita_misura'
)

# pz without specific conversion → treat as kg
pz_mask = recipes_cleaned_std['unita_misura'] == 'pz'
recipes_cleaned_std.loc[pz_mask, 'unita_misura'] = 'kg'
print(f'pz rows converted to kg: {pz_mask.sum()}')
print(f'recipes_cleaned_std: {recipes_cleaned_std.shape}')
display(recipes_cleaned_std.head())

In [ ]:
# Split 'sale e pepe' → 'sale fino' + 'pepe nero' (quantities halved)
sale_pepe_mask = recipes_cleaned_std['ingrediente'].str.strip() == 'sale e pepe'
print(f'sale e pepe rows found: {sale_pepe_mask.sum()}')

if sale_pepe_mask.sum() > 0:
    orig = recipes_cleaned_std.loc[sale_pepe_mask].copy()

    sale_rows = orig.copy()
    sale_rows['ingrediente']            = 'sale fino'
    sale_rows['quantita_per_porzione']  = (orig['quantita_per_porzione'] / 2).round(4)
    sale_rows['costo_stimato_eur']      = (orig['costo_stimato_eur'] / 2).round(4)

    pepe_rows = orig.copy()
    pepe_rows['ingrediente']            = 'pepe nero'
    pepe_rows['quantita_per_porzione']  = (orig['quantita_per_porzione'] / 2).round(4)
    pepe_rows['costo_stimato_eur']      = (orig['costo_stimato_eur'] / 2).round(4)

    recipes_cleaned_std = pd.concat(
        [recipes_cleaned_std[~sale_pepe_mask], sale_rows, pepe_rows],
        ignore_index=True
    )
    print(f'Split done — sale fino: {len(sale_rows)} row(s), pepe nero: {len(pepe_rows)} row(s)')

print(f'recipes_cleaned_std after split: {recipes_cleaned_std.shape}')
display(recipes_cleaned_std[recipes_cleaned_std['ingrediente'].isin(['sale fino', 'pepe nero'])])

---
# 3. OUTLIER FIXING
Pattern: `detect (multiplier=4)` → food mask → `→ NaN` → benchmark fill.

Applied to: **inventory** (`valore_unitario`) and **invoices** (`prezzo_unitario`).

### 3.1 Inventory — valore_unitario

In [ ]:
inv_out_mask  = ut.outliers_auto_detection(inventory_cleaned_std, multiplier=4)
inv_food_mask = ~inventory_cleaned_std['categoria'].isin(BEV_CATEGORIES)
inv_fix_mask  = inv_out_mask & inv_food_mask

print(f'Outlier rows to fix: {inv_fix_mask.sum()}')
display(
    inventory_cleaned_std.loc[inv_fix_mask, ['prodotto', 'categoria', 'unita_misura', 'valore_unitario']]
    .sort_values('valore_unitario', ascending=False)
    .drop_duplicates(subset=['valore_unitario'])
)

In [ ]:
# Step 1 — set outlier prices to NaN
inventory_cleaned_std.loc[inv_fix_mask, 'valore_unitario'] = np.nan

# Step 2 — count ALL NaN before imputing
nan_unit  = inventory_cleaned_std['valore_unitario'].isna().sum()
nan_total = inventory_cleaned_std['valore_totale_eur'].isna().sum() \
            if 'valore_totale_eur' in inventory_cleaned_std.columns else 'col not present'
print(f'NaN before imputing — valore_unitario: {nan_unit}  |  valore_totale_eur: {nan_total}')

# Step 3 — impute valore_unitario via nome_standard → benchmark_dict
benchmark_fill = inventory_cleaned_std['nome_standard'].map(benchmark_dict)
inventory_cleaned_std['valore_unitario'] = inventory_cleaned_std['valore_unitario'].fillna(benchmark_fill)

# Step 4 — recalculate valore_totale_eur = qty_fisica * valore_unitario
inventory_cleaned_std['valore_totale_eur'] = (
    inventory_cleaned_std['qty_fisica'] * inventory_cleaned_std['valore_unitario']
).round(2)

# Step 5 — report NaN still remaining
nan_unit_after  = inventory_cleaned_std['valore_unitario'].isna().sum()
nan_total_after = inventory_cleaned_std['valore_totale_eur'].isna().sum()
print(f'NaN after  imputing — valore_unitario: {nan_unit_after}  |  valore_totale_eur: {nan_total_after}')

if nan_unit_after > 0:
    print('\nRows still missing valore_unitario (no benchmark match):')
    display(
        inventory_cleaned_std.loc[inventory_cleaned_std['valore_unitario'].isna(),
                                  ['prodotto', 'nome_standard', 'categoria', 'unita_misura', 'qty_fisica']]
    )

print(f'\ninventory_cleaned_std: {inventory_cleaned_std.shape}')

### 3.2 Invoices — prezzo_unitario

In [ ]:
inv2_out_mask  = ut.outliers_auto_detection(invoices_cleaned_std, multiplier=4)
inv2_food_mask = ~invoices_cleaned_std['categoria'].isin(BEV_CATEGORIES)
inv2_fix_mask  = inv2_out_mask & inv2_food_mask

print(f'Outlier rows to fix: {inv2_fix_mask.sum()}')
display(
    invoices_cleaned_std.loc[inv2_fix_mask, ['descrizione_prodotto', 'categoria', 'unita_misura', 'prezzo_unitario']]
    .sort_values('prezzo_unitario', ascending=False)
    .drop_duplicates(subset=['prezzo_unitario'])
)

In [ ]:
invoices_cleaned_std.loc[inv2_fix_mask, 'prezzo_unitario'] = np.nan

benchmark_fill2 = invoices_cleaned_std['nome_standard'].map(benchmark_dict)
invoices_cleaned_std['prezzo_unitario'] = invoices_cleaned_std['prezzo_unitario'].fillna(benchmark_fill2)

# Recalculate importo_totale_pulito on corrected prices
invoices_cleaned_std['importo_totale_pulito'] = (
    invoices_cleaned_std['prezzo_unitario'] * invoices_cleaned_std['quantita']
).round(2)

# NaN Handling Rule — print and stop, no auto-drop
# AWAIT GIOVANNI APPROVAL
nan_after = invoices_cleaned_std['prezzo_unitario'].isna().sum()
print(f'NaN left after benchmark fill — prezzo_unitario: {nan_after}')

if nan_after > 0:
    print('\nRows still missing prezzo_unitario (no benchmark match):')
    display(
        invoices_cleaned_std.loc[invoices_cleaned_std['prezzo_unitario'].isna(),
                                 ['descrizione_prodotto', 'nome_standard', 'categoria', 'unita_misura', 'quantita']]
        .sort_values('categoria')
    )
else:
    print('All prices imputed — no NaN remaining.')

print(f'\ninvoices_cleaned_std: {invoices_cleaned_std.shape}')

---
# 4. AUGMENTATION
6 dishes present in POS but missing from recipe book → hardcoded dict → `pd.concat`.

> Source: `_extra/ebitda_pipeline.ipynb` — cell 73

In [ ]:
data_augmentation = [
    # --- cotoletta alla milanese ---
    {'nome_piatto': 'cotoletta alla milanese', 'ingrediente': 'fesa di vitello',        'quantita_per_porzione': 0.250, 'unita_misura': 'kg', 'costo_stimato_eur': 4.50},
    {'nome_piatto': 'cotoletta alla milanese', 'ingrediente': 'pangrattato',             'quantita_per_porzione': 0.060, 'unita_misura': 'kg', 'costo_stimato_eur': 0.15},
    {'nome_piatto': 'cotoletta alla milanese', 'ingrediente': 'uovo',                    'quantita_per_porzione': 0.060, 'unita_misura': 'kg', 'costo_stimato_eur': 0.25},
    {'nome_piatto': 'cotoletta alla milanese', 'ingrediente': 'farina',                  'quantita_per_porzione': 0.030, 'unita_misura': 'kg', 'costo_stimato_eur': 0.05},
    {'nome_piatto': 'cotoletta alla milanese', 'ingrediente': 'burro chiarificato',      'quantita_per_porzione': 0.050, 'unita_misura': 'kg', 'costo_stimato_eur': 0.60},
    # --- tagliata di manzo ---
    {'nome_piatto': 'tagliata di manzo',       'ingrediente': 'controfiletto manzo',     'quantita_per_porzione': 0.300, 'unita_misura': 'kg', 'costo_stimato_eur': 7.50},
    {'nome_piatto': 'tagliata di manzo',       'ingrediente': 'rucola',                  'quantita_per_porzione': 0.050, 'unita_misura': 'kg', 'costo_stimato_eur': 0.40},
    {'nome_piatto': 'tagliata di manzo',       'ingrediente': 'grana',                   'quantita_per_porzione': 0.040, 'unita_misura': 'kg', 'costo_stimato_eur': 0.40},
    # --- pappardelle al cinghiale (ragu exploded) ---
    # was: 'ragu di cinghiale' 0.150 kg → split into real ingredients (tot = 0.150 kg)
    {'nome_piatto': 'pappardelle al cinghiale','ingrediente': 'pappardelle fresche',     'quantita_per_porzione': 0.125, 'unita_misura': 'kg', 'costo_stimato_eur': 0.70},
    {'nome_piatto': 'pappardelle al cinghiale','ingrediente': 'cinghiale macinato',      'quantita_per_porzione': 0.100, 'unita_misura': 'kg', 'costo_stimato_eur': 2.00},
    {'nome_piatto': 'pappardelle al cinghiale','ingrediente': 'pomodoro pelato',         'quantita_per_porzione': 0.030, 'unita_misura': 'kg', 'costo_stimato_eur': 0.10},
    {'nome_piatto': 'pappardelle al cinghiale','ingrediente': 'cipolla',                 'quantita_per_porzione': 0.010, 'unita_misura': 'kg', 'costo_stimato_eur': 0.02},
    {'nome_piatto': 'pappardelle al cinghiale','ingrediente': 'carote',                  'quantita_per_porzione': 0.010, 'unita_misura': 'kg', 'costo_stimato_eur': 0.02},
    # --- tagliere di salumi (selezione exploded + sottoli → giardiniera) ---
    # was: 'selezione salumi toscani' 0.150 kg → split into real ingredients (tot = 0.150 kg)
    {'nome_piatto': 'tagliere di salumi',      'ingrediente': 'prosciutto crudo toscano','quantita_per_porzione': 0.060, 'unita_misura': 'kg', 'costo_stimato_eur': 1.80},
    {'nome_piatto': 'tagliere di salumi',      'ingrediente': 'finocchiona',             'quantita_per_porzione': 0.050, 'unita_misura': 'kg', 'costo_stimato_eur': 1.00},
    {'nome_piatto': 'tagliere di salumi',      'ingrediente': 'lardo di colonnata',      'quantita_per_porzione': 0.040, 'unita_misura': 'kg', 'costo_stimato_eur': 0.80},
    # was: 'sottoli' → renamed to canonical ingredient name
    {'nome_piatto': 'tagliere di salumi',      'ingrediente': 'giardiniera di verdure',  'quantita_per_porzione': 0.050, 'unita_misura': 'kg', 'costo_stimato_eur': 0.25},
    {'nome_piatto': 'tagliere di salumi',      'ingrediente': 'pane',                    'quantita_per_porzione': 0.100, 'unita_misura': 'kg', 'costo_stimato_eur': 0.30},
    # --- bruschette miste ---
    {'nome_piatto': 'bruschette miste',        'ingrediente': 'pane casereccio',         'quantita_per_porzione': 0.100, 'unita_misura': 'kg', 'costo_stimato_eur': 0.30},
    {'nome_piatto': 'bruschette miste',        'ingrediente': 'pomodoro',                'quantita_per_porzione': 0.150, 'unita_misura': 'kg', 'costo_stimato_eur': 1.20},
    # --- ribollita toscana (verdure exploded) ---
    # was: 'verdure' 0.200 kg → split into real ingredients (tot = 0.200 kg)
    {'nome_piatto': 'ribollita toscana',       'ingrediente': 'cavolo nero',             'quantita_per_porzione': 0.080, 'unita_misura': 'kg', 'costo_stimato_eur': 0.20},
    {'nome_piatto': 'ribollita toscana',       'ingrediente': 'carote',                  'quantita_per_porzione': 0.060, 'unita_misura': 'kg', 'costo_stimato_eur': 0.07},
    {'nome_piatto': 'ribollita toscana',       'ingrediente': 'cipolla',                 'quantita_per_porzione': 0.040, 'unita_misura': 'kg', 'costo_stimato_eur': 0.04},
    {'nome_piatto': 'ribollita toscana',       'ingrediente': 'sedano',                  'quantita_per_porzione': 0.020, 'unita_misura': 'kg', 'costo_stimato_eur': 0.03},
    {'nome_piatto': 'ribollita toscana',       'ingrediente': 'fagioli borlotti secchi', 'quantita_per_porzione': 0.150, 'unita_misura': 'kg', 'costo_stimato_eur': 0.42},
    {'nome_piatto': 'ribollita toscana',       'ingrediente': 'pane raffermo',           'quantita_per_porzione': 0.100, 'unita_misura': 'kg', 'costo_stimato_eur': 0.20},
    {'nome_piatto': 'ribollita toscana',       'ingrediente': 'olio evo',                'quantita_per_porzione': 0.020, 'unita_misura': 'kg', 'costo_stimato_eur': 0.30},
]

recipes_aug_df      = pd.DataFrame(data_augmentation)
recipes_cleaned_aug = pd.concat([recipes_cleaned_std, recipes_aug_df], ignore_index=True)

print(f'Original rows : {len(recipes_cleaned_std)}')
print(f'Added rows    : {len(recipes_aug_df)}')
print(f'Total         : {len(recipes_cleaned_aug)} rows — {recipes_cleaned_aug["nome_piatto"].nunique()} unique dishes')

In [ ]:
# AWAIT GIOVANNI APPROVAL
# Review augmented recipe list above before fuzzy match runs.
# Check: ingredient names, quantities, costo_stimato_eur look correct?
print('--- Augmented recipe preview (approve before running fuzzy match) ---')
display(recipes_cleaned_aug.groupby('nome_piatto')[['ingrediente', 'quantita_per_porzione', 'costo_stimato_eur']].apply(lambda x: x.reset_index(drop=True)))

In [ ]:
# Fuzzy match augmented recipes + benchmark price fill
# Run this cell only after approving the preview above
# NOTE: benchmark reloaded from CSV at setup — new rows included automatically on fresh run
fuzzy_map_recipes = ut.get_best_match(
    recipes_cleaned_aug['ingrediente'],
    benchmark_cleaned['Ingrediente_Standard'],
    threshold=70
)
recipes_cleaned_aug['nome_standard'] = recipes_cleaned_aug['ingrediente'].map(fuzzy_map_recipes)

recipes_cleaned_aug['prezzo_benchmark_eur'] = (
    recipes_cleaned_aug['nome_standard'].map(benchmark_dict)
    .fillna(recipes_cleaned_aug['costo_stimato_eur'])
)

# Step 4 — report NaN remaining + AWAIT GIOVANNI APPROVAL
unmatched_rec = recipes_cleaned_aug.loc[recipes_cleaned_aug['nome_standard'].isna()]
print(f'Unmatched ingredients after updated benchmark: {unmatched_rec["ingrediente"].nunique()}')

if len(unmatched_rec) > 0:
    print('\n--- NaN nome_standard — recipes ---')
    print('Categoria breakdown:')
    # recipes have no 'categoria' column — group by nome_piatto instead
    display(unmatched_rec['nome_piatto'].value_counts(dropna=False))
    print('\nFull unmatched list:')
    display(unmatched_rec.drop_duplicates(subset=['ingrediente'])[['nome_piatto', 'ingrediente', 'quantita_per_porzione', 'costo_stimato_eur']])
else:
    print('All ingredients matched — no NaN remaining.')

# AWAIT GIOVANNI APPROVAL
print('\nReview unmatched above. Approve before proceeding to Section 3 (Outlier Fixing).')
display(recipes_cleaned_aug[['nome_piatto', 'ingrediente', 'nome_standard', 'prezzo_benchmark_eur']].head(15))

---
## Final state

| Variable | Ready for |
|----------|-----------|
| `benchmark_cleaned` | price lookup |
| `benchmark_dict` | shared price reference |
| `inventory_cleaned_std` | ETL join |
| `sales_pos_cleaned` | ETL join |
| `invoices_cleaned_std` | ETL join |
| `recipes_cleaned_aug` | costing, EBITDA |

---
## Coverage check — sales_pos vs recipes
Read-only. Fuzzy match sales_pos.item_name → recipes will be done in SQL (Notebook 03).
This cell only flags dishes sold but missing from the recipe book.

In [ ]:
from thefuzz import process, fuzz

known_dishes = recipes_cleaned_aug['nome_piatto'].str.lower().unique().tolist()
pos_items    = sales_pos_cleaned['item_name'].dropna().str.lower().unique()

matched, unmatched = [], []
for item in pos_items:
    best = process.extractOne(item, known_dishes, scorer=fuzz.token_set_ratio)
    if best and best[1] >= 70:
        matched.append({'item_name': item, 'matched_dish': best[0], 'score': best[1]})
    else:
        unmatched.append({'item_name': item, 'best_guess': best[0] if best else None, 'score': best[1] if best else 0})

print(f'Unique dishes in sales_pos : {len(pos_items)}')
print(f'Matched to recipe book     : {len(matched)}  ({round(len(matched)/len(pos_items)*100)}%)')
print(f'No recipe match (< 70)     : {len(unmatched)}')

if unmatched:
    print('\n--- Dishes sold with no recipe match (to handle in Notebook 03) ---')
    display(pd.DataFrame(unmatched).sort_values('score', ascending=False))

> **Note — unmatched item_names:** The items with no recipe match are neither dishes nor single ingredients — they are POS line entries of a different nature (e.g. service charges, covers, promotions, or catch-all categories). They do not belong to the recipe book and cannot be costed. They will be excluded in the ETL pipeline (Notebook 03) during the food cost join.

In [ ]:
for name, df in [
    ('benchmark_cleaned',     benchmark_cleaned),
    ('inventory_cleaned_std', inventory_cleaned_std),
    ('sales_pos_cleaned',     sales_pos_cleaned),
    ('invoices_cleaned_std',  invoices_cleaned_std),
    ('recipes_cleaned_aug',   recipes_cleaned_aug),
]:
    nan_pct = round(df.isna().mean().mean() * 100, 1)
    print(f'  {name:<30} {df.shape[0]:>5} rows  {df.shape[1]:>2} cols  avg_missing: {nan_pct}%')